[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session2/notebooks/S2_NB02_pipeline_pretraitement.ipynb)

# S2-NB02 — Construire un pipeline de prétraitement audio

**Module 1 — Session 2 : Traitement du signal audio** · Atelier 2B · 40 min · Ticket `M1-S2-05`

## 🎯 Objectifs
- **Diagnostiquer** un enregistrement : niveau, saturation, composante continue, rapport signal/bruit (SNR).
- Construire **votre** fonction `mon_preprocess()` étape par étape : normalisation, coupe des silences, détection de
  parole (VAD), segmentation, (débruitage).
- **Segmenter** un enregistrement radio réel de près de 3 minutes en énoncés exploitables.
- **Mesurer** si le débruitage aide vraiment la reconnaissance vocale.
- Extraire les **features** d'entrée des modèles (log-Mel, MFCC) et préparer vos fichiers pour la session 3.

## 🧭 Comment travailler
| Badge | Pour qui | Attendu |
|---|---|---|
| 🟢 Essentiel | Tout le monde | Exécuter, régler, écouter, interpréter (≈ 25 min) |
| 🔵 Intermédiaire | À l'aise en Python | Compléter le code `✍️ À COMPLÉTER` |
| 🔴 Avancé | Experts ML | Défis ouverts |

## 0. Préchauffage

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from m1utils.audio import load_audio, play, save_audio
from m1utils.data import load_manifest
from m1utils.features import log_mel, mfcc
from m1utils.preprocess import (PreprocessConfig, add_noise, dbfs, denoise, detect_clipping, estimate_snr,
                                frame_energy_db, normalize_loudness, normalize_peak, normalize_rms, preprocess,
                                remove_dc, split_on_segments, trim_silence, vad_segments)
from m1utils.viz import plot_mel, plot_waveform

# 📝 PARAMÈTRE : votre dossier TP1 (laisser tel quel pour utiliser les exemples)
MON_DOSSIER_TP1 = "/content/drive/MyDrive/TP1_nom"

demo = load_manifest("demo_s1")
long = load_manifest("long_recording_s2")          # témoignage radio wolof (Kallaama), 168 s, + version bruitée
mes_fichiers = sorted(Path(MON_DOSSIER_TP1, "audio").glob("*.*")) if Path(MON_DOSSIER_TP1, "audio").exists() else []
fichiers = mes_fichiers or [Path(p) for p in demo.abspath]
print(f"{len(fichiers)} fichiers ({'vos enregistrements' if mes_fichiers else 'exemples du cours'}) ; "
      f"enregistrement long : {long.duration_s.iloc[0]:.0f} s")

## 1. Diagnostic 🟢
Pour chaque fichier : niveau crête et niveau moyen (RMS) en **dBFS** (0 dBFS = pleine échelle ; tout est négatif),
taux de **saturation**, composante continue (**DC offset**, la moyenne du signal, normalement ≈ 0) et **SNR** estimé.

In [ ]:
def diagnostic(chemin):
    y, sr = load_audio(chemin)
    return {"fichier": Path(chemin).name[:40], "durée_s": len(y) / sr, "crête_dBFS": dbfs(y, "peak"),
            "RMS_dBFS": dbfs(y, "rms"), "saturation_%": 100 * detect_clipping(y), "DC": float(np.mean(y)),
            "SNR_dB": estimate_snr(y, sr)}


diag = pd.DataFrame([diagnostic(f) for f in fichiers] + [diagnostic(p) for p in long.abspath])
display(diag.round(2))

**❓ Question :** quels fichiers ont un niveau très faible (RMS < −35 dBFS) ? un SNR médiocre (< 15 dB) ?
La version `_bruit10dB` est-elle bien diagnostiquée ?

✍️ *Votre réponse :* …

## 2. Normaliser le niveau 🔵
Objectif : que tous les fichiers aient un niveau comparable. Trois méthodes :
- **crête** : le maximum à −1 dBFS (simple, mais sensible à un seul « pic ») ;
- **RMS** : l'énergie moyenne à −20 dBFS ;
- **LUFS** : la sonie perçue (norme ITU-R BS.1770, celle de la radio et des plateformes).

Implémentez la normalisation crête : multiplier le signal pour que son maximum absolu vaille `10^(peak_dbfs/20)`.

In [ ]:
def ma_normalisation_pic(y, peak_dbfs=-1.0):
    gain = ...   # ✍️ À COMPLÉTER
    return (y * gain).astype(np.float32)


y, sr = load_audio(fichiers[0])
assert np.isclose(np.max(np.abs(ma_normalisation_pic(y))), 10 ** (-1 / 20), atol=1e-4), "Le pic doit valoir -1 dBFS"
assert np.allclose(ma_normalisation_pic(y), normalize_peak(y, -1.0), atol=1e-5), "Différent de la référence m1utils"
print("✅ Normalisation crête correcte")
for nom, z in [("original", y), ("crête -1 dBFS", normalize_peak(y)), ("RMS -20 dBFS", normalize_rms(y)),
               ("LUFS -23", normalize_loudness(y, sr))]:
    play(z, sr, label=f"{nom} — RMS {dbfs(z, 'rms'):.1f} dBFS", normalize=False)

## 3. Couper les silences de début et de fin 🟢
`trim_silence` retire ce qui est à plus de `top_db` dB sous le maximum. Réglez le curseur : trop **bas**, on coupe
des sons faibles (fin de mot, consonne sourde) ; trop **haut**, on garde du bruit.

In [ ]:
def explorer_trim(top_db=30):
    yt, (debut, fin) = trim_silence(y, top_db=top_db)
    ax = plot_waveform(y, sr, segments=[(debut / sr, fin / sr)],
                       title=f"top_db = {top_db} : on garde {len(yt) / sr:.2f} s sur {len(y) / sr:.2f} s")
    plt.show()


widgets.interact(explorer_trim, top_db=widgets.IntSlider(30, 10, 60, 5));

## 4. Détecter la parole (VAD) 🔵
Une **détection d'activité vocale** (*Voice Activity Detection*, VAD) indique où se trouve la parole. Version la plus
simple : calculer l'énergie de trames de 30 ms et comparer à un **seuil**.

Complétez `mon_vad` : renvoie la liste des segments (début_s, fin_s) où l'énergie dépasse `seuil_db`.

In [ ]:
def mon_vad(y, sr, seuil_db=-35.0):
    temps, energie = frame_energy_db(y, sr, frame_s=0.03, hop_s=0.01)
    actif = energie > seuil_db
    segments, debut = [], None
    for t, a in zip(temps, actif):
        ...  # ✍️ À COMPLÉTER : ouvrir un segment quand a devient vrai, le fermer quand a redevient faux
    if debut is not None:
        segments.append((debut, temps[-1]))
    return segments


extrait, sr = load_audio(long.abspath.iloc[0])
extrait = normalize_peak(extrait[: 30 * sr])  # 30 premières secondes de l'émission radio
segments_maison = mon_vad(extrait, sr)
segments_silero = vad_segments(extrait, sr, method="silero")
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
plot_waveform(extrait, sr, ax=axes[0], segments=segments_maison, title=f"VAD maison : {len(segments_maison)} segments")
plot_waveform(extrait, sr, ax=axes[1], segments=segments_silero, title=f"VAD Silero (réseau de neurones) : {len(segments_silero)} segments")
plt.tight_layout()
plt.show()

**❓ Question :** combien de segments chaque méthode trouve-t-elle ? Laquelle « hache » la parole en petits morceaux ?
Que manque-t-il au VAD maison (indice : regardez `m1utils.preprocess._energy_vad`) ?

✍️ *Votre réponse :* …

## 5. Segmenter un long enregistrement 🟢
Les modèles traitent mal les longs fichiers (Whisper : 30 s maximum ; mémoire GPU). On découpe l'émission entière en
énoncés grâce au VAD, en re-coupant les segments trop longs au point le plus calme.

In [ ]:
emission, sr = load_audio(long.abspath.iloc[0])
emission = normalize_peak(emission)
MIN_SILENCE_S = 0.3   # 📝 à régler (exercice 5.1)
segments = vad_segments(emission, sr, method="silero", min_silence_s=MIN_SILENCE_S)
morceaux = split_on_segments(emission, segments, sr, pad_s=0.1, max_segment_s=15)
durees = np.array([len(m) / sr for m in morceaux])
print(f"{len(morceaux)} énoncés · durée médiane {np.median(durees):.1f} s · min {durees.min():.1f} s · max {durees.max():.1f} s")
plt.hist(durees, bins=20)
plt.xlabel("durée de l'énoncé (s)")
plt.ylabel("nombre")
plt.title("Distribution des durées après segmentation")
plt.show()
for m in morceaux[:3]:
    play(m, sr)

### Exercice 5.1 🔵 — Régler la segmentation
Pour l'ASR, on vise des énoncés de **2 à 15 s**. Essayez `MIN_SILENCE_S` = 0,2 ; 0,5 ; 1,0 et comptez la proportion
d'énoncés dans [2, 15] s. Quel réglage retenez-vous ?

In [ ]:
...  # ✍️ À COMPLÉTER

✍️ *Votre réponse :* …

## 6. Débruiter… avec prudence 🟢
Le débruitage par **soustraction spectrale** estime le spectre du bruit et le retire. Agréable à l'oreille — mais
aide-t-il la **machine** ? Mesurons-le : on ajoute du bruit à 10 énoncés propres, on débruite, et on fait transcrire
les trois versions par Simba-H (CER : taux d'erreur sur les caractères).

In [ ]:
from m1utils.asr import load_asr, unload
from m1utils.metrics import cer
from m1utils.text import normalize_text

test = load_manifest("testset_s3").query("domain == 'lu'").head(10)
modele = load_asr("simba-h")
SNR_DB = 10   # 📝 essayez 20, 10, 5, 0
scores = {"propre": [], "bruité": [], "débruité": []}
for ligne in test.itertuples():
    propre, sr = load_audio(ligne.abspath)
    bruite = add_noise(propre, SNR_DB, seed=1)
    for nom, signal in [("propre", propre), ("bruité", bruite), ("débruité", denoise(bruite, sr))]:
        hyp = modele(signal).text
        scores[nom].append(cer(normalize_text(ligne.text, ligne.lang), normalize_text(hyp, ligne.lang)))
unload(modele)
display(pd.DataFrame(scores).mean().rename(f"CER moyen (SNR {SNR_DB} dB)").round(3).to_frame())
play(bruite, sr, label="Bruité")
play(denoise(bruite, sr), sr, label="Débruité")

**❓ Question :** le débruitage ramène-t-il le CER au niveau de l'audio propre ? Que conseillez-vous pour un projet réel ?

✍️ *Votre réponse :* …

## 7. Assembler `mon_preprocess` 🔵
Enchaînez : mono → 16 kHz → retrait DC → normalisation crête → coupe des silences → **marge de 0,2 s de silence**
remise de chaque côté. Comparez avec la référence `m1utils.preprocess.preprocess` (durée identique à ± 5 %).

⚠️ Pourquoi remettre une marge ? **Mesure du cours** : couper au ras de la parole fait passer le WER de Simba-H de
0,27 à 0,63 sur le fongbe lu (le modèle se met à écrire des tons !) ; avec 0,2 s de marge, 0,285. Le modèle a appris
sur des enregistrements entourés de silence : on reproduit les **conditions d'entraînement**.

In [ ]:
def mon_preprocess(chemin, peak_dbfs=-1.0, top_db=30, marge_s=0.2):
    y, sr = ...   # ✍️ À COMPLÉTER : charger en mono 16 kHz
    ...           # ✍️ À COMPLÉTER : retrait DC, normalisation crête, trim, marge (np.pad)
    return y, sr


for f in fichiers[:5]:
    mien, sr = mon_preprocess(f)
    brut, sr0 = load_audio(f, sr=None)
    reference, _ = preprocess(brut, sr0, PreprocessConfig())
    ecart = abs(len(mien) - len(reference)) / len(reference)
    assert ecart < 0.05, f"{f.name} : durée trop différente de la référence ({ecart:.0%})"
print("✅ mon_preprocess cohérent avec la référence")

## 8. Ce que « voient » les modèles : log-Mel et MFCC 🟢
- **Whisper** reçoit un spectrogramme **log-Mel** : 80 bandes sur une échelle perceptive (Mel), une trame / 10 ms.
- Les systèmes classiques (HMM-GMM) utilisaient les **MFCC** : 13 coefficients résumant l'enveloppe spectrale.
- **wav2vec2 / HuBERT / Simba-H** reçoivent le **signal brut** et apprennent leurs propres features (une trame / 20 ms).

In [ ]:
y, sr = mon_preprocess(fichiers[0])
M, C = log_mel(y, sr), mfcc(y, sr)
print(f"Signal : {len(y)} échantillons ({len(y) / sr:.2f} s) → log-Mel {M.shape} · MFCC {C.shape}")
plot_mel(y, sr, title="Ce que Whisper « voit »")
plt.show()

### Exercice 8.1 🔵 — Combien de trames ?
Whisper traite des fenêtres de 30 s. Combien de trames log-Mel cela fait-il ? Et pour wav2vec2 (20 ms par trame en
sortie) ? Vérifiez sur un signal de 30 s de silence.

In [ ]:
trames_whisper = ...   # ✍️ À COMPLÉTER : log_mel(np.zeros(30 * 16_000)).shape[1]
print(f"Whisper : {trames_whisper} trames (≈ 3 000) · wav2vec2 : ≈ {trames_wav2vec2} trames")

## 9. Préparer mon corpus pour la session 3 🟢
On applique `mon_preprocess` à tous vos fichiers TP1 et on écrit un **rapport qualité**. Ces fichiers propres et ce
rapport sont les entrées du TP2 et de la session 3.

In [ ]:
sortie = Path(MON_DOSSIER_TP1) / "clean" if mes_fichiers else Path("/tmp/m1_clean")
lignes = []
for f in fichiers:
    brut, sr0 = load_audio(f, sr=None)
    propre, sr = mon_preprocess(f)
    chemin = save_audio(sortie / f"{f.stem}.wav", propre, sr)
    lignes.append({"id": f.stem, "path": str(chemin), "duration_s": round(len(propre) / sr, 3), "sr": sr,
                   "snr_db": round(estimate_snr(propre, sr), 1), "clipping_ratio": round(detect_clipping(brut), 4),
                   "décision": "garder" if detect_clipping(brut) < 0.001 else "réenregistrer"})
rapport = pd.DataFrame(lignes)
rapport.to_csv(sortie / "rapport_qualite.csv", index=False)
display(rapport)
print(f"→ {len(rapport)} fichiers propres dans {sortie}")

## 10. Défi 🔴 — Quel prétraitement pour quel modèle ?
Comparez, sur les 10 énoncés de la partie 6 **bruités à 10 dB**, le CER de Simba-H pour trois configurations :
(a) aucun traitement ; (b) normalisation LUFS ; (c) LUFS + débruitage léger (`prop_decrease=0.5`). Conclusion ?

In [ ]:
...  # ✍️ À COMPLÉTER

## ✅ Bilan
- On **diagnostique** avant de traiter : niveau (dBFS), saturation (irréversible !), SNR.
- Pipeline de référence : mono → 16 kHz → retrait DC → normalisation → coupe des silences → (VAD + segmentation).
- Un **VAD neuronal** (Silero) est bien plus fiable qu'un seuil d'énergie ; la segmentation se **règle** et se mesure.
- Le **débruitage** plaît à l'oreille mais aide peu (voire nuit à) la machine : mesurez, et préférez de bons
  enregistrements + l'augmentation de données.
- Whisper voit un **log-Mel** (100 trames/s, 3 000 pour 30 s) ; wav2vec2/HuBERT voient le **signal brut**.

**Prochaine étape :** TP2 — analyse de vos spectrogrammes et nettoyage complet de votre corpus.